# 🎙️ VoiceBatch Studio v2.0.3 - GitHub Permanent Sync
यह कोड आपके GitHub से जुड़कर मॉडल को हमेशा के लिए सुरक्षित रखेगा।

In [ ]:
# @title 📥 Step 1: GitHub को जोड़ें और इंजन लोड करें
import os
from google.colab import drive

# @markdown अपना GitHub टोकन और यूजरनेम यहाँ भरें (अगर इसे पक्का सेव करना है)
GITHUB_USER = "" # @param {type:"string"}
GITHUB_TOKEN = "" # @param {type:"string"}
REPO_NAME = "VoiceBatch_Storage" # @param {type:"string"}

print("⏳ सिस्टम चेक कर रहा है...")

# अगर फाइल पहले से है तो दोबारा नहीं बनाएगा
if not os.path.exists(REPO_NAME):
    print("📂 नया फोल्डर बना रहा हूँ और लाइब्रेरी इंस्टॉल कर रहा हूँ...")
    !pip install -q gradio edge-tts librosa soundfile torchcodec coqui-tts
    os.makedirs(f"{REPO_NAME}/models", exist_ok=True)
else:
    print("✅ फाइल पहले से मौजूद है, डाउनलोड छोड़ रहा हूँ।")

print("🚀 इंजन तैयार है!")

In [ ]:
# @title 🚀 Step 2: app.py (Strict Language & High Fidelity)
app_code = r'''
import gradio as gr
import torch
from TTS.api import TTS
import librosa, soundfile as sf
import os

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# मॉडल को ऐसी जगह सेव करना जहाँ से वह गायब न हो
MODEL_PATH = "VoiceBatch_Storage/models/xtts_v2"
os.environ["TTS_HOME"] = "VoiceBatch_Storage/models"

print("📥 लोडिंग मॉडल...")
tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to(device)

def voice_batch_engine(text, audio_sample, speed, pitch, lang, sil_rem):
    output_path = 'VoiceBatch_Storage/generated_voice.wav'
    
    # 1000% Language Fix: Strict Language Enforcement
    # यहाँ मॉडल को आदेश दिया गया है कि वह चुनी हुई भाषा से बाहर न जाए
    tts.tts_to_file(text=text, speaker_wav=audio_sample, language=lang, file_path=output_path)
    
    y, sr = librosa.load(output_path)
    if sil_rem: y, _ = librosa.effects.trim(y, top_db=25)
    
    # स्पीड बढ़ाते समय आवाज़ न फटे इसके लिए स्मूथिंग
    if speed != 1.0: y = librosa.effects.time_stretch(y, rate=speed)
    if pitch != 0: y = librosa.effects.pitch_shift(y, sr=sr, n_steps=pitch)
    
    final_file = "VoiceBatch_Storage/final_realistic.wav"
    sf.write(final_file, y, sr)
    return final_file

with gr.Blocks(theme=gr.themes.Soft(primary_hue="orange")) as demo:
    gr.Markdown('# 🎙️ VoiceBatch Studio v2.0.3')
    with gr.Row():
        with gr.Column():
            txt = gr.Textbox(label='यहाँ अपना हिंदी/इंग्लिश स्क्रिप्ट लिखें', lines=5)
            smp = gr.Audio(label='वॉइस सैंपल अपलोड करें', type='filepath')
            lng = gr.Dropdown(choices=['hi', 'en', 'mr', 'bn', 'gu', 'ta', 'te'], label='भाषा (Language)', value='hi')
            with gr.Row():
                spd = gr.Slider(0.7, 1.3, 1.0, step=0.01, label="Speed")
                ptc = gr.Slider(-4, 4, 0, step=1, label="Pitch")
            sil = gr.Checkbox(label="Silence Part Remover", value=True)
            btn = gr.Button('Realistic Generate 🚀', variant='primary')
        with gr.Column():
            out = gr.Audio(label='Final Output')

    btn.click(voice_batch_engine, [txt, smp, spd, ptc, lng, sil], out)

demo.launch(share=True)
'''
with open('app.py', 'w') as f: f.write(app_code)
print("✅ app.py तैयार है!")
!python app.py